<a href="https://colab.research.google.com/github/dokturdro/ragvsfinetune/blob/main/ragvsfinetune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Install Dependencies
We'll install Unsloth for fast fine-tuning and LangChain for the RAG components.

In [1]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install langchain langchain-community sentence-transformers faiss-gpu pypdf
!pip install rouge-score


  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-6gl36yui/unsloth_1a611c87048f4a60b097d65ee5436a41
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-6gl36yui/unsloth_1a611c87048f4a60b097d65ee5436a41
  Resolved https://github.com/unslothai/unsloth.git to commit 6c1cc50e84bcb097a432e9577b81d8870f42aa8a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of trl to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidanc

### 2. Load the Model and Tokenizer
Using Unsloth to load Qwen 2.5 in 4-bit quantization for Colab compatibility.

In [2]:
import torch
from unsloth import FastLanguageModel

model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

### 3. Data Preparation

In [3]:
long_supplements_text_v2 = """
# **Dietary Supplements: A Comprehensive Guide to Benefits, Mechanisms, and Clinical Evidence**

Dietary supplements encompass a broad category of products intended to complement the diet and provide nutrients, botanicals, or other substances that may be missing or not consumed in sufficient quantities. Their efficacy, safety, and interactions are subjects of extensive scientific inquiry, with varying levels of robust clinical evidence for each compound.

## **Foundational Micronutrients**

### **Vitamin D3 (Cholecalciferol)**

*   **Benefits:** Essential for calcium homeostasis and bone mineralization, reducing osteoporosis risk. Plays critical roles in immune system modulation, exhibiting immunoregulatory effects that may lower the incidence of autoimmune diseases like multiple sclerosis and type 1 diabetes. Emerging research highlights its impact on cardiovascular health, mood regulation, and cognitive function. Meta-analyses consistently link adequate Vitamin D status with reduced all-cause mortality. Recent double-blind, placebo-controlled trials continue to elucidate its nuanced impact on various physiological systems, including inflammation and cellular growth.
*   **Mechanisms:** Functions as a prohormone, binding to the Vitamin D Receptor (VDR) to regulate gene expression in numerous tissues. Influences innate and adaptive immunity, cell differentiation, and neurotrophic factor synthesis.
*   **Forms & Dosage:** Cholecalciferol is the most common and bioavailable form. Typical prophylactic doses range from 2,000-5,000 IU daily; therapeutic doses for deficiency correction can exceed 10,000 IU under strict medical supervision, with monitoring of serum 25(OH)D levels.
*   **Concerns:** Hypervitaminosis D is rare but severe, leading to hypercalcemia, nephrolithiasis (kidney stones), and soft tissue calcification. Requires careful monitoring, especially when combined with high calcium intake. Interactions with corticosteroids and antiepileptic drugs.

### **Magnesium**

*   **Benefits:** A ubiquitous cation involved in over 300 enzymatic reactions, crucial for ATP synthesis, DNA/RNA replication, protein synthesis, nerve impulse transmission, muscle contraction, blood glucose control, and blood pressure regulation. Reduces muscle cramps, alleviates restless leg syndrome, and supports cardiovascular health. Emerging evidence suggests benefits for migraine prophylaxis and reduction of anxiety/stress. Contributes to neurotransmitter function, influencing mood and sleep quality.
*   **Mechanisms:** Acts as a co-factor for various enzymes, stabilizes ATP, modulates NMDA receptors, and influences smooth muscle tone. Deficient intake is widespread in modern diets.
*   **Forms & Dosage:** Diverse forms target specific needs: Magnesium Citrate (laxative, high bioavailability), Magnesium L-Threonate (patented for blood-brain barrier penetration, investigated for cognitive enhancement and memory), Magnesium Glycinate (high absorption, minimal GI upset, calming for sleep and anxiety), Magnesium Malate (energy support, muscle pain), Magnesium Oxide (poor bioavailability, primarily for constipation). Daily intake typically 200-400 mg elemental magnesium.
*   **Concerns:** Excessive oral intake can cause osmotic diarrhea, nausea, and abdominal cramping. Severe hypermagnesemia from oral supplements is rare in individuals with normal renal function. Intravenous administration requires stringent medical oversight due to risks of hypotension, bradycardia, and respiratory depression.

### **Zinc**

*   **Benefits:** Essential for immune system function, protein synthesis, wound healing, DNA synthesis, cell division, and maintaining proper taste and smell perception. Supports male reproductive health and hormone balance. Critical for enzymatic function in hundreds of metalloenzymes. Short-term use (e.g., lozenges) may reduce duration of common cold symptoms.
*   **Mechanisms:** Functions as a catalytic, structural, and regulatory ion. Directly participates in immune cell development and function.
*   **Forms & Dosage:** Zinc Picolinate, Gluconate, Citrate are common forms with good bioavailability. Typical daily dosages range from 15-30 mg elemental zinc. Short-term cold relief dosages can be higher.
*   **Concerns:** Chronic or high-dose zinc monotherapy (e.g., >40 mg/day for extended periods) can induce copper deficiency due to competitive absorption, leading to symptoms like anemia, neurological dysfunction, and impaired immune response. A balanced zinc-to-copper ratio (e.g., 10:1 to 15:1) is crucial for long-term supplementation.

### **Vitamin C (Ascorbic Acid)**

*   **Benefits:** Potent antioxidant, crucial for collagen synthesis (skin, bones, cartilage), immune function, and neurotransmitter biosynthesis. Aids in iron absorption. May reduce the duration and severity of the common cold, though not prevent it. Supports adrenal function under stress.
*   **Mechanisms:** Acts as an electron donor, protecting cells from oxidative damage. Required co-factor for hydroxylase enzymes involved in collagen formation.
*   **Forms & Dosage:** Ascorbic acid is the most common. Liposomal Vitamin C offers enhanced bioavailability. Daily dosages range from 500-2000 mg.
*   **Concerns:** High doses can cause gastrointestinal upset (diarrhea, nausea) and increase risk of kidney stones in susceptible individuals. May interact with anticoagulant medications.

### **B Vitamins (B-Complex)**

*   **Benefits:** A group of eight water-soluble vitamins crucial for cellular metabolism, energy production, neurological function, and DNA synthesis. Specific benefits include: **Vitamin B12 (Cobalamin)**: Essential for red blood cell formation, neurological function, and DNA synthesis; crucial for vegans/vegetarians. **Folate (B9)**: Vital for cell growth and DNA formation, critical during pregnancy to prevent neural tube defects. **Vitamin B6 (Pyridoxine)**: Involved in amino acid metabolism, neurotransmitter synthesis (serotonin, dopamine), and immune function. **Thiamine (B1)**, **Riboflavin (B2)**, **Niacin (B3)**, **Pantothenic Acid (B5)**, and **Biotin (B7)**: Play key roles in energy metabolism.
*   **Mechanisms:** Function as coenzymes in various metabolic pathways.
*   **Forms & Dosage:** Bioavailable forms like methylcobalamin (B12) and methylfolate (B9) are often preferred. Dosages vary widely.
*   **Concerns:** Generally safe; excessive B6 can cause peripheral neuropathy. High doses of niacin can cause flushing. Interactions with certain medications (e.g., isoniazid).

### **Vitamin K2 (Menaquinone)**

*   **Benefits:** Crucial for calcium utilization, directing calcium to bones and teeth, and away from arteries and soft tissues. Works synergistically with Vitamin D3 to support bone mineral density and cardiovascular health by activating Matrix Gla Protein (MGP) which inhibits arterial calcification. Emerging research on cognitive function and nerve health.
*   **Mechanisms:** Activates Vitamin K-dependent proteins, including osteocalcin (for bone mineralization) and MGP (for vascular health).
*   **Forms & Dosage:** MK-4 and MK-7 are common forms; MK-7 has a longer half-life. Dosages typically 100-300 mcg daily.
*   **Concerns:** Contraindicated with anticoagulant medications (e.g., warfarin) due to interference with blood clotting. Consult a physician.

## **Essential Fatty Acids**

### **Omega-3 Fatty Acids (EPA & DHA)**

*   **Benefits:** Potent anti-inflammatory properties, cardiovascular protection (reducing triglycerides, blood pressure, and risk of arrhythmias), neurocognitive benefits (brain development, memory, mood), and support for eye health (macular degeneration). Major clinical trials, such as REDUCE-IT (high-dose EPA), have demonstrated significant reductions in major adverse cardiovascular events in high-risk patients. Observational studies suggest benefits for mood disorders (e.g., depression), rheumatoid arthritis, and inflammatory bowel diseases.
*   **Mechanisms:** Precursors to anti-inflammatory resolvins and protectins. Modulate cell membrane fluidity and signaling pathways.
*   **Sources & Forms:** Fish oil (triglyceride form for better absorption), krill oil (phospholipid form), and sustainable algal oil (vegan source of DHA, sometimes EPA). Optimal EPA:DHA ratios are debated, with higher EPA often favored for anti-inflammatory effects.
*   **Concerns:** Low-quality products are prone to oxidative rancidity, negating benefits and potentially introducing harmful byproducts. Heavy metal contamination (mercury, PCBs) is a concern with non-purified fish oils. High doses (>3g/day) can prolong bleeding time, requiring caution with anticoagulants or prior to surgery. Mild GI upset is possible.

## **Gut Health Modulators**

### **Probiotics**

*   **Benefits:** Live microorganisms that, when administered in adequate amounts, confer a health benefit to the host. Support a healthy gut microbiome, improve digestion, enhance immune function, and may alleviate symptoms of gastrointestinal disorders like IBS, IBD, and antibiotic-associated diarrhea. Specific strains have shown efficacy in areas such as mood (gut-brain axis), skin health, and weight management. Strain-specific efficacy is paramount.
*   **Mechanisms:** Modulate gut microbiota composition, produce short-chain fatty acids (e.g., butyrate), reinforce gut barrier integrity, compete with pathogens, and modulate local and systemic immune responses.
*   **Key Strains:** *Lactobacillus rhamnosus GG (LGG)* (antibiotic-associated diarrhea, pediatric gut health), *Bifidobacterium lactis (BB-12)* (bowel regularity, immune modulation), *Lactobacillus plantarum 299v* (IBS symptoms), *Saccharomyces boulardii* (yeast probiotic, traveler's diarrhea).
*   **Concerns:** Efficacy is highly context-dependent, varying with the individual's baseline microbiome, health status, and specific product. Side effects can include gas, bloating, or mild GI upset initially. Immunocompromised individuals should use with caution.

### **Prebiotics**

*   **Benefits:** Non-digestible food ingredients that selectively stimulate the growth and/or activity of beneficial bacteria (e.g., Bifidobacteria, Lactobacilli) in the colon. Improve gut microbiome diversity, promote regularity, and enhance mineral absorption. Often synergistically combined with probiotics (symbiotics).
*   **Mechanisms:** Fermented by gut bacteria to produce short-chain fatty acids (SCFAs) like butyrate, which nourish colonocytes, reduce inflammation, and may have systemic metabolic benefits.
*   **Sources:** Inulin, Fructooligosaccharides (FOS), Galactooligosaccharides (GOS), resistant starch. Found naturally in garlic, onions, leeks, bananas, asparagus.
*   **Concerns:** Can cause gas and bloating, especially at high doses or in sensitive individuals. Start with low doses and gradually increase.

## **Herbal Adaptogens & Botanicals**

### **Ashwagandha (Withania somnifera)**

*   **Benefits:** Ayurvedic adaptogenic herb, renowned for its anxiolytic (anxiety-reducing) and stress-modulating properties. Helps the body adapt to various stressors, reduces serum cortisol levels, improves subjective stress resilience, and enhances sleep quality. May improve physical performance, fertility, and cognitive function. Multiple placebo-controlled trials support its efficacy.
*   **Mechanisms:** Modulates the hypothalamic-pituitary-adrenal (HPA) axis, influences GABAergic and serotonergic signaling, and exhibits antioxidant effects. Active compounds are withanolides.
*   **Forms & Dosage:** Root extract standardized for withanolides (e.g., KSM-66, Sensoril). Typical daily dosage 300-600 mg of extract.
*   **Concerns:** Generally well-tolerated. May cause mild drowsiness or GI upset. Contraindicated in pregnancy and with certain thyroid conditions or autoimmune diseases. Potential interactions with sedatives or immunosuppressants.

### **Rhodiola Rosea**

*   **Benefits:** Adaptogenic herb used for combating fatigue, improving cognitive performance under stressful conditions, and enhancing physical endurance. Supports mental clarity and reduces symptoms of burnout. May improve athletic performance and mood.
*   **Mechanisms:** Influences neurotransmitter levels (serotonin, dopamine, norepinephrine), modulates HPA axis, and protects against oxidative stress. Active compounds are rosavins and salidrosides.
*   **Forms & Dosage:** Root extract standardized for rosavins and salidrosides. Typical dosage 200-600 mg daily.
*   **Concerns:** Generally safe; may cause insomnia or irritability in sensitive individuals. Avoid in bipolar disorder or with MAO inhibitors.

### **Panax Ginseng**

*   **Benefits:** Classic adaptogen, studied for its immunomodulatory, neuroprotective, anti-fatigue, and cognitive-enhancing properties. May improve physical stamina, reduce stress, and support overall vitality. Used traditionally for centuries.
*   **Mechanisms:** Active compounds, ginsenosides, interact with various physiological systems, including nitric oxide synthesis, anti-inflammatory pathways, and neurotransmitter release.
*   **Forms & Dosage:** Standardized extract for ginsenosides. Dosage varies significantly.
*   **Concerns:** Can interact with anticoagulants, antidiabetic drugs, and stimulants. May cause insomnia or GI upset. Not recommended for long-term continuous use.

### **Turmeric (Curcuma Longa) / Curcumin**

*   **Benefits:** Potent anti-inflammatory and antioxidant properties. May alleviate symptoms of chronic inflammatory conditions (e.g., osteoarthritis, inflammatory bowel disease), support joint health, and improve cognitive function. Investigated for its chemopreventive properties.
*   **Mechanisms:** Curcumin, the active compound, inhibits numerous inflammatory pathways (e.g., NF-κB, COX-2, LOX) and scavenges free radicals.
*   **Forms & Dosage:** Curcumin has notoriously low oral bioavailability. Formulations often include piperine (black pepper extract) to enhance absorption, or use advanced delivery systems (e.g., liposomal, phytosome formulations). Doses typically range from 500-1500 mg of curcuminoids.
*   **Concerns:** Generally safe. High doses may cause GI upset. Can interact with blood thinners and antidiabetic drugs. Caution in individuals with gallstones.

### **St. John's Wort (Hypericum perforatum)**

*   **Benefits:** Well-established botanical for treating mild to moderate depression. Numerous clinical trials demonstrate efficacy comparable to selective serotonin reuptake inhibitors (SSRIs), with a generally favorable side effect profile for appropriate cases.
*   **Mechanisms:** Active compounds (hypericin, hyperforin) are thought to inhibit the reuptake of serotonin, norepinephrine, and dopamine. Also influences GABA and glutamate receptors.
*   **Forms & Dosage:** Standardized extract for hyperforin or hypericin. Typical dosage 300 mg (0.3% hypericin) three times daily.
*   **Concerns:** **Significant drug-herb interactions** due to potent induction of cytochrome P450 enzymes (CYP3A4, CYP2C9, CYP1A2) and P-glycoprotein. Interacts with oral contraceptives, antidepressants, antiretrovirals, digoxin, warfarin, and many other medications. **Requires medical consultation due to potential for serotonin syndrome when combined with other serotonergic agents.** Avoid in severe depression or bipolar disorder.

### **Milk Thistle (Silybum marianum)**

*   **Benefits:** Traditionally used as a hepatoprotective agent for liver support and detoxification. Its active component, silymarin, is a powerful antioxidant and anti-inflammatory, protecting liver cells from damage and promoting regeneration. May be beneficial in alcoholic liver disease, viral hepatitis, and toxin-induced liver injury.
*   **Mechanisms:** Silymarin stabilizes liver cell membranes, enhances antioxidant defenses (glutathione), and inhibits inflammatory pathways.
*   **Forms & Dosage:** Standardized extract for silymarin. Typical dosage 200-400 mg silymarin daily.
*   **Concerns:** Generally well-tolerated. Mild laxative effect possible. Possible interactions with CYP3A4 substrates.

### **Berberine**

*   **Benefits:** An isoquinoline alkaloid from various plants (e.g., Goldenseal, Barberry). Exhibits potent hypoglycemic (blood sugar-lowering) and lipid-lowering effects, often compared to metformin for type 2 diabetes and statins for dyslipidemia. May improve insulin sensitivity and support cardiovascular health.
*   **Mechanisms:** Activates AMP-activated protein kinase (AMPK), inhibits gluconeogenesis, upregulates LDL receptors, and modulates gut microbiota.
*   **Forms & Dosage:** Berberine HCl is common. Typical dosage 500 mg 2-3 times daily.
*   **Concerns:** Can cause significant gastrointestinal side effects (nausea, diarrhea, constipation, cramping), limiting compliance. Interacts with CYP3A4 and P-glycoprotein, potentially affecting metabolism of other drugs. Caution with antidiabetic medications due to risk of hypoglycemia.

## **Amino Acids & Derivatives**

### **Creatine Monohydrate**

*   **Benefits:** One of the most rigorously studied and effective ergogenic aids. Consistently demonstrates benefits for high-intensity exercise performance, muscle mass accretion, strength gains, and faster recovery. Also shows neuroprotective and cognitive-enhancing effects (e.g., memory, executive function) in specific populations (e.g., vegetarians, sleep-deprived individuals). Improves cellular energy production.
*   **Mechanisms:** Increases intramuscular phosphocreatine stores, which rapidly regenerate ATP during short bursts of high-intensity activity. Enhances satellite cell activity and gene expression related to muscle growth.
*   **Forms & Dosage:** Creatine monohydrate is the gold standard. Loading phase (5g x 4 times/day for 5-7 days) followed by maintenance (3-5g/day) is common, though a gradual non-loading approach is also effective.
*   **Concerns:** Excellent safety profile. May cause mild GI distress or water retention in some individuals. No evidence of adverse effects on kidney function in healthy individuals, but caution in pre-existing renal disease.

### **N-Acetyl Cysteine (NAC)**

*   **Benefits:** A stable precursor to the master antioxidant glutathione. Functions as a potent antioxidant, detoxifying agent, and mucolytic (thinning mucus). Used medically for acetaminophen overdose. Increasingly investigated for its potential therapeutic roles in psychiatric indications (e.g., OCD, substance use disorders, schizophrenia), respiratory conditions (COPD, bronchitis), and liver support (non-alcoholic fatty liver disease).
*   **Mechanisms:** Replenishes glutathione stores, scavenges reactive oxygen species, and modulates glutamatergic neurotransmission.
*   **Forms & Dosage:** N-Acetyl Cysteine. Dosages vary widely based on indication, typically 600-1800 mg daily.
*   **Concerns:** Generally safe. May cause GI upset, nausea. Can interact with blood pressure medications. Has a characteristic sulfurous odor.

### **L-Theanine**

*   **Benefits:** An amino acid analog found primarily in green tea. Promotes a state of relaxed alertness without sedation, by increasing alpha brain waves and modulating neurotransmitters. Reduces anxiety and stress, improves focus and attention. Often synergistically combined with caffeine to enhance cognitive performance, reduce jitters, and smooth out the stimulant experience.
*   **Mechanisms:** Increases GABA, serotonin, and dopamine levels. Directly affects brain wave activity.
*   **Forms & Dosage:** L-Theanine. Typical dosage 100-200 mg.
*   **Concerns:** Generally safe. No significant side effects or interactions reported.

### **Branched-Chain Amino Acids (BCAAs: Leucine, Isoleucine, Valine)**

*   **Benefits:** Popular among athletes for muscle protein synthesis, reducing exercise-induced muscle damage, and attenuating fatigue during prolonged exercise. Leucine is key for initiating mTOR pathway signaling for muscle growth.
*   **Mechanisms:** Directly metabolized in muscle tissue. Serve as building blocks for protein and energy source during exercise.
*   **Forms & Dosage:** Leucine, isoleucine, valine, typically in a 2:1:1 ratio. Daily dosages 5-10g.
*   **Concerns:** While popular, their standalone efficacy compared to sufficient intake of complete protein (which provides all essential amino acids) is debated outside of specific contexts (e.g., fasted training, catabolic states). May disrupt amino acid balance with long-term high-dose use. Potential negative interaction with L-tryptophan absorption into the brain.

### **L-Arginine / L-Citrulline**

*   **Benefits:** Both are precursors to nitric oxide (NO), a vasodilator that relaxes blood vessels, improving blood flow. L-Arginine is often used for cardiovascular health, erectile dysfunction, and exercise performance. L-Citrulline bypasses liver metabolism, converting more efficiently to L-Arginine, and is often preferred for more sustained NO production and greater benefits for exercise performance, blood pressure reduction, and post-exercise muscle soreness.
*   **Mechanisms:** L-Arginine is directly converted to NO by nitric oxide synthase. L-Citrulline is converted to L-Arginine in the kidneys.
*   **Forms & Dosage:** L-Arginine HCl; L-Citrulline or Citrulline Malate (often preferred). Dosages 3-6g for L-Arginine, 6-8g for L-Citrulline/Citrulline Malate.
*   **Concerns:** Can cause GI upset (diarrhea, nausea) at high doses. Interactions with blood pressure medications and erectile dysfunction drugs (synergistic hypotensive effects). Contraindicated in herpes simplex due to potential for outbreak.

### **Glycine**

*   **Benefits:** A simple amino acid that acts as an inhibitory neurotransmitter. Promotes relaxation, improves sleep quality, supports collagen synthesis, and has hepatoprotective effects. Involved in glutathione synthesis.
*   **Mechanisms:** Binds to glycine receptors in the brain, promoting sleep. Important for protein structure and detoxification pathways.
*   **Forms & Dosage:** Glycine powder. Typical dosages 3-9g before bed for sleep.
*   **Concerns:** Generally safe. Very few side effects.

## **Antioxidants & Co-factors**

### **Coenzyme Q10 (CoQ10) / Ubiquinol**

*   **Benefits:** An endogenously synthesized, fat-soluble antioxidant essential for mitochondrial electron transport and ATP production. Supports cellular energy, cardiovascular health (e.g., heart failure, blood pressure), and exerts powerful antioxidant effects. Commonly supplemented by individuals on statin medications to mitigate statin-induced myopathy due to statins inhibiting CoQ10 synthesis, though trial results for this specific indication are mixed. Ubiquinol, the reduced form, offers enhanced bioavailability, particularly in older individuals or those with impaired absorption.
*   **Mechanisms:** Crucial component of the electron transport chain. Scavenges free radicals, protecting mitochondrial membranes.
*   **Forms & Dosage:** Ubiquinone (standard form) or Ubiquinol (more bioavailable). Typical dosage 100-300 mg daily, higher for specific conditions.
*   **Concerns:** Generally safe. May cause mild GI upset. Can interact with warfarin (antagonistic effect) and chemotherapy drugs.

### **Alpha-Lipoic Acid (ALA)**

*   **Benefits:** A powerful antioxidant and mitochondrial co-factor involved in energy metabolism. Unique in being both water- and fat-soluble, allowing it to work throughout the body. Studied for its benefits in diabetic neuropathy (improving nerve function and reducing pain), metabolic syndrome (insulin-sensitizing properties), and cognitive health. Regenerates other antioxidants like Vitamin C, Vitamin E, and glutathione.
*   **Mechanisms:** Co-factor for mitochondrial enzymes (e.g., pyruvate dehydrogenase). Direct free radical scavenger and chelator of heavy metals.
*   **Forms & Dosage:** R-ALA (biologically active form) or racemic mixture (R/S-ALA). Typical dosage 300-600 mg daily.
*   **Concerns:** Generally safe. May cause mild GI upset. Caution with antidiabetic medications due to potential for hypoglycemia. Can interact with thyroid hormones and chemotherapy.

### **Resveratrol**

*   **Benefits:** A polyphenol found in red wine, grapes, and Japanese knotweed. Noted for its antioxidant, anti-inflammatory, and potential anti-aging properties. Investigated for cardiovascular benefits, neuroprotection, and glucose metabolism. Often studied for its potential to activate sirtuins, a class of proteins linked to longevity.
*   **Mechanisms:** Activates sirtuins (SIRT1), modulates inflammatory pathways, and protects against oxidative stress.
*   **Forms & Dosage:** Trans-resveratrol is the active form. Typical dosage 100-500 mg daily.
*   **Concerns:** Human data on its broad anti-aging and health claims are less robust than in vitro and animal studies, partly due to poor bioavailability. May interact with blood thinners. Mild GI upset possible.

### **Astaxanthin**

*   **Benefits:** A potent carotenoid antioxidant, responsible for the red-pink color in salmon and flamingos. Offers superior antioxidant activity compared to Vitamin E and Beta-carotene. Benefits include eye health (reducing eye strain, protecting against UV damage), skin health (improving elasticity, reducing wrinkles), cardiovascular protection, and exercise performance/recovery by reducing oxidative stress.
*   **Mechanisms:** Embeds in cell membranes, offering broad-spectrum antioxidant protection. Reduces inflammation.
*   **Forms & Dosage:** Natural astaxanthin from *Haematococcus pluvialis* algae. Typical dosage 4-12 mg daily.
*   **Concerns:** Generally safe. Very few reported side effects.

## **Joint & Bone Health**

### **Glucosamine Sulfate & Chondroitin Sulfate**

*   **Benefits:** Frequently used in combination for joint health, particularly for mitigating symptoms of osteoarthritis. May help rebuild cartilage, reduce joint pain, and improve mobility. Clinical evidence for efficacy is mixed and varies with the severity of the condition and specific formulation. Some meta-analyses suggest benefits for pain reduction and improved function, especially with high-quality, pharmaceutical-grade preparations.
*   **Mechanisms:** Glucosamine is a precursor to glycosaminoglycans, a major component of cartilage. Chondroitin helps maintain cartilage structure and fluid retention.
*   **Forms & Dosage:** Glucosamine sulfate is generally considered more effective than glucosamine HCl. Typical dosage 1500 mg glucosamine and 800-1200 mg chondroitin daily.
*   **Concerns:** Generally safe. May cause mild GI upset. Potential interactions with anticoagulants (warfarin). Contraindicated in shellfish allergies (for some glucosamine sources).

### **Methylsulfonylmethane (MSM)**

*   **Benefits:** An organic sulfur compound. Often used for joint pain, inflammation, and exercise recovery. Supports connective tissue health (cartilage, skin, hair, nails). May reduce muscle damage and oxidative stress after exercise.
*   **Mechanisms:** Provides sulfur for cartilage formation. Exerts anti-inflammatory and antioxidant effects.
*   **Forms & Dosage:** MSM powder or capsules. Typical dosage 1-6g daily.
*   **Concerns:** Generally safe. Mild GI upset possible.

### **Collagen Peptides**

*   **Benefits:** Protein found abundantly in connective tissues (skin, bones, cartilage, tendons). Supplementation with hydrolyzed collagen peptides (Type I, II, III) supports skin elasticity, hydration, and reduces wrinkle depth. Also promotes joint health (reducing pain in osteoarthritis), bone mineral density, and gut integrity. May strengthen hair and nails.
*   **Mechanisms:** Provides amino acid building blocks (glycine, proline, hydroxyproline) for endogenous collagen synthesis. Signals to fibroblasts to produce more collagen.
*   **Forms & Dosage:** Hydrolyzed collagen (peptides) from bovine, marine, or chicken sources. Type I & III for skin/bones; Type II for joints. Typical dosage 10-20g daily.
*   **Concerns:** Generally safe. May cause mild GI upset. Source matters for potential allergens.

## **Sleep & Mood Support**

### **Melatonin**

*   **Benefits:** A neurohormone primarily produced by the pineal gland, regulating circadian rhythms and sleep-wake cycles. Widely used for sleep initiation (reducing sleep onset latency), combating jet lag, and regulating sleep patterns in shift workers. May also have antioxidant properties.
*   **Mechanisms:** Signals to the brain that it is nighttime, promoting drowsiness.
*   **Forms & Dosage:** Melatonin. Dosages typically 0.5-5 mg 30-60 minutes before bed. Lower doses are often more effective for sleep regulation.
*   **Concerns:** Can cause daytime drowsiness, vivid dreams, or headaches. May interact with blood thinners, immunosuppressants, and antidiabetic drugs. Not recommended for long-term daily use without medical guidance.

### **5-Hydroxytryptophan (5-HTP)**

*   **Benefits:** A direct precursor to serotonin, a neurotransmitter involved in mood, sleep, appetite, and pain sensation. Used for mood enhancement, improving sleep quality, and appetite suppression. May be beneficial for mild depression, anxiety, and fibromyalgia.
*   **Mechanisms:** Bypasses the rate-limiting step of tryptophan hydroxylase in serotonin synthesis.
*   **Forms & Dosage:** 5-HTP. Typical dosage 50-200 mg daily.
*   **Concerns:** Can cause GI upset (nausea, vomiting). **Caution is advised due to potential for serotonin syndrome when combined with other serotonergic medications** (SSRIs, MAOIs, tricyclics). Avoid in pregnancy and with certain medical conditions (e.g., Carcinoid Syndrome).

## **Other Notable Supplements**

### **Green Tea Extract (EGCG)**

*   **Benefits:** Rich in catechins, particularly epigallocatechin gallate (EGCG), a powerful antioxidant. Supports cardiovascular health, metabolic health (fat oxidation, thermogenesis), and provides neuroprotection. May have chemopreventive properties.
*   **Mechanisms:** Potent antioxidant, modulates various signaling pathways involved in metabolism and cellular protection.
*   **Forms & Dosage:** Standardized extract for EGCG. Typical dosage 200-500 mg EGCG daily.
*   **Concerns:** High doses, especially on an empty stomach, can cause liver toxicity in susceptible individuals. Contains caffeine unless decaffeinated. Interactions with warfarin.

### **Spirulina & Chlorella**

*   **Benefits:** Nutrient-dense blue-green algae, touted as superfoods. Rich in protein, vitamins (B12 in spirulina, though often inactive analog), minerals, and antioxidants. Support detoxification (heavy metal chelation, especially chlorella), immune function, and gut health. May improve cholesterol levels and blood pressure.
*   **Mechanisms:** High chlorophyll content, potent antioxidants, and unique phytonutrients contribute to their effects.
*   **Forms & Dosage:** Powder or tablets. Dosages typically 3-10g daily.
*   **Concerns:** Source quality is crucial to avoid contamination with heavy metals or toxins (e.g., microcystins). May interact with immunosuppressants or anticoagulants.

### **Beta-Alanine**

*   **Benefits:** An amino acid that combines with histidine to form carnosine in muscle tissue. Carnosine buffers lactic acid, reducing muscle fatigue and improving high-intensity exercise performance (e.g., sprints, weightlifting sets). Enhances muscular endurance.
*   **Mechanisms:** Increases intramuscular carnosine concentrations.
*   **Forms & Dosage:** Beta-Alanine. Typical dosage 3.2-6.4g daily, often split into smaller doses.
*   **Concerns:** Can cause paresthesia (tingling sensation on skin), a harmless but sometimes uncomfortable side effect, particularly at higher doses. Can be mitigated by lower split doses or sustained-release formulations.

### **Lion's Mane Mushroom (Hericium erinaceus)**

*   **Benefits:** A medicinal mushroom known for its neurotrophic properties. May support cognitive function (memory, focus), nerve regeneration, and mood. Traditionally used in Eastern medicine for digestive health and as a tonic.
*   **Mechanisms:** Contains hericenones and erinacines, compounds that stimulate nerve growth factor (NGF) synthesis.
*   **Forms & Dosage:** Extract from fruiting body or mycelium. Dosage varies.
*   **Concerns:** Generally safe. Very few reported side effects. More human clinical trials are needed to confirm all benefits.

## **General Considerations for Supplementation**

The complex interplay of these bioactives underscores the critical importance of personalized approaches, evidence-based decision-making, and professional guidance in navigating the realm of nutraceuticals. While the potential for therapeutic intervention is vast, the distinction between anecdotal claims and rigorously peer-reviewed clinical data remains paramount for informed health choices. Considerations of dosage, delivery matrix, contraindications, potential polypharmacy interactions, individual variability in response, and product quality (third-party testing for purity and potency) are critical for safe, effective, and ethical supplementation strategies in clinical practice and personal wellness journeys.
"""

with open('supplements_data.txt', 'w') as f:
    f.write(long_supplements_text_v2)

print("Saved updated, extended supplements data to 'supplements_data.txt'")

Saved updated, extended supplements data to 'supplements_data.txt'


In [19]:
from datasets import Dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

with open('supplements_data.txt', 'r') as f:
    raw_text = f.read()

recursive_text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    separators=[
        "\n\n",
        "\n",
        " ",
        "",
    ]
)

recursive_docs = recursive_text_splitter.create_documents([raw_text])

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(recursive_docs, embeddings)

print(f"Created {len(recursive_docs)} document chunks using RecursiveCharacterTextSplitter.")

training_data = [
    {
        "instruction": "What are the primary benefits and forms of Magnesium?",
        "input": "",
        "output": "Magnesium is crucial for over 300 enzymatic reactions, ATP synthesis, and nerve function. Common forms include Magnesium Citrate (laxative, high bioavailability), Magnesium L-Threonate (blood-brain barrier penetration for cognitive support), and Magnesium Glycinate (calming for sleep and anxiety)."
    },
    {
        "instruction": "Summarize the concerns regarding high-dose Zinc supplementation.",
        "input": "",
        "output": "Chronic or high-dose zinc monotherapy (>40 mg/day) can induce copper deficiency due to competitive absorption. This can lead to anemia, neurological dysfunction, and impaired immune response. A balanced zinc-to-copper ratio is crucial for long-term use."
    },
    {
        "instruction": "What are the potential drug interactions with St. John's Wort?",
        "input": "",
        "output": "St. John's Wort has significant drug-herb interactions because it potently induces cytochrome P450 enzymes (like CYP3A4) and P-glycoprotein. It can interact with oral contraceptives, antidepressants, antiretrovirals, digoxin, and warfarin. Combining it with other serotonergic agents risks serotonin syndrome."
    },
    {
        "instruction": "Describe the benefits of Omega-3 Fatty Acids.",
        "input": "",
        "output": "Omega-3 Fatty Acids (EPA and DHA) offer potent anti-inflammatory properties, cardiovascular protection, and neurocognitive benefits. They are associated with reduced triglycerides, blood pressure, and risk of arrhythmias, as well as support for eye health and mood disorders."
    }
]

dataset = Dataset.from_list(training_data)

def format_prompt(example):
    return {"text": f"Instruction: {example['instruction']}\nInput: {example['input']}\nOutput: {example['output']}"}

dataset = dataset.map(format_prompt)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Created 468 document chunks using RecursiveCharacterTextSplitter.


Map:   0%|          | 0/4 [00:00<?, ? examples/s]

### Advanced RAG: Two-Stage Retrieval (Vector Search + Reranking)

In [20]:
from sentence_transformers import CrossEncoder

print("Loading CrossEncoder for reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def retrieve_with_reranker(query, top_k=10, final_k=2):
    initial_docs = vectorstore.similarity_search(query, k=top_k)
    pairs = [[query, doc.page_content] for doc in initial_docs]
    scores = cross_encoder.predict(pairs)
    doc_score_pairs = list(zip(initial_docs, scores))
    doc_score_pairs.sort(key=lambda x: x[1], reverse=True)

    return [doc for doc, score in doc_score_pairs[:final_k]]

print("Advanced RAG setup complete.")

Loading CrossEncoder for reranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Advanced RAG setup complete.


In [21]:
def retrieve_multi_query(original_query, query_variations, k_per_query=2):

    queries_to_search = [original_query] + query_variations
    all_results = []

    for q in queries_to_search:
        docs = vectorstore.similarity_search(q, k=k_per_query)
        all_results.extend(docs)

    unique_docs = []
    seen_content = set()
    for doc in all_results:
        if doc.page_content not in seen_content:
            seen_content.add(doc.page_content)
            unique_docs.append(doc)

    print(f"Multi-Query: Searched {len(queries_to_search)} variations, found {len(unique_docs)} unique chunks.")
    return unique_docs

base_query = "What forms of Magnesium exist?"
mocked_llm_variations = [
    "Types of magnesium supplements available",
    "Which magnesium is best for absorption and sleep?"
]

demo_docs = retrieve_multi_query(base_query, mocked_llm_variations)
print(f"Top result preview: {demo_docs[0].page_content[:100]}...")

Multi-Query: Searched 3 variations, found 3 unique chunks.
Top result preview: 200-400 mg elemental magnesium....


### 4. Fine-tuning the Model

In [22]:
from trl import SFTTrainer
from transformers import TrainingArguments

model = FastLanguageModel.get_peft_model(
    model, r=16, target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=16, lora_dropout=0, bias="none", use_gradient_checkpointing="unsloth",
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=dataset,
    dataset_text_field="text", max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        max_steps=60, learning_rate=2e-4, logging_steps=1, output_dir="outputs",
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    ),
)
trainer.train()

Unsloth: Already have LoRA adapters! We shall skip this step.


Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 10,092,544 of 7,625,709,056 (0.13% trained)


Step,Training Loss
1,0.018596
2,0.111502
3,0.042761
4,0.038236
5,0.030676
6,0.026373
7,0.021631
8,0.019972
9,0.018579
10,0.018748


TrainOutput(global_step=60, training_loss=0.02006664052605629, metrics={'train_runtime': 124.6497, 'train_samples_per_second': 3.851, 'train_steps_per_second': 0.481, 'total_flos': 930065640591360.0, 'train_loss': 0.02006664052605629})

### Advanced Fine-Tuning: DoRA (Weight-Decomposed Low-Rank Adaptation)


In [27]:
from trl import SFTTrainer
from transformers import TrainingArguments
import torch
import gc
from unsloth import FastLanguageModel

if 'model' in globals():
    del model
if 'trainer' in globals():
    del trainer
gc.collect()
torch.cuda.empty_cache()

model_dora_base, tokenizer_dora = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit
)

model_dora = FastLanguageModel.get_peft_model(
    model_dora_base,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    use_dora=True,
)

trainer_dora = SFTTrainer(
    model=model_dora,
    tokenizer=tokenizer_dora,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        max_steps=60,
        learning_rate=2e-4,
        logging_steps=10,
        output_dir="outputs_dora",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

trainer_dora.train()

==((====))==  Unsloth 2026.9.12: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth: transformers renamed `push_to_hub_token` to `hub_token`. Forwarding your value to `hub_token` - update your code when convenient. If you also passed `hub_token` as None, that is its default here and cannot be distinguished from leaving it unset, so `push_to_hub_token` was used; drop `push_to_hub_token` to keep it.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/4 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,760,768 of 7,657,377,280 (0.55% trained)


Step,Training Loss
10,1.039457
20,0.046448
30,0.018603
40,0.017444
50,0.017078
60,0.017033


TrainOutput(global_step=60, training_loss=0.19267732451359432, metrics={'train_runtime': 612.1888, 'train_samples_per_second': 0.784, 'train_steps_per_second': 0.098, 'total_flos': 934225325150208.0, 'train_loss': 0.19267732451359432})

### 5. Compare RAG vs Fine-tuning
Define inference functions and measure results.

In [29]:
FastLanguageModel.for_inference(model_dora)

def generate_text(prompt):
    inputs = tokenizer_dora([prompt], return_tensors="pt").to("cuda")
    outputs = model_dora.generate(**inputs, max_new_tokens=150, use_cache=True)
    return tokenizer_dora.batch_decode(outputs, skip_special_tokens=True)[0].split("Output:")[-1].strip()

def get_basic_rag_response(query):
    docs = vectorstore.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"Context: {context}\nInstruction: {query}\nInput: \nOutput:"
    return generate_text(prompt)

def get_advanced_rag_response(query):
    docs = retrieve_with_reranker(query, top_k=10, final_k=2)
    context = "\n".join([doc.page_content for doc in docs])
    prompt = f"Context: {context}\nInstruction: {query}\nInput: \nOutput:"
    return generate_text(prompt)

def get_lora_ft_response(query):
    prompt = f"Instruction: {query}\nInput: \nOutput:"
    return generate_text(prompt)

def get_dora_ft_response(query):
    prompt = f"Instruction: {query}\nInput: \nOutput:"
    return generate_text(prompt)

print("Real inference functions ready.")

Real inference functions ready.


### Evaluation Framework

In [30]:
from rouge_score import rouge_scorer
from sentence_transformers import SentenceTransformer, util

eval_model = SentenceTransformer('all-MiniLM-L6-v2')
scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

def evaluate_responses(query, ground_truth, responses):
    print(f"\n{'='*70}")
    print(f"Query: {query}")
    print(f"Ground Truth: {ground_truth}\n")

    gt_emb = eval_model.encode(ground_truth)

    for method_name, response_text in responses.items():
        print(f"{method_name} Response: {response_text[:100]}...")
        rouge = scorer.score(ground_truth, response_text)
        emb = eval_model.encode(response_text)
        sim = util.cos_sim(gt_emb, emb).item()
        print(f"  -> ROUGE-L F1: {rouge['rougeL'].fmeasure:.4f} | Semantic Sim: {sim:.4f}\n")
    print(f"{'='*70}\n")

eval_queries = [
    (
        "What are the primary benefits and forms of Magnesium?",
        "Magnesium is crucial for over 300 enzymatic reactions, ATP synthesis, and nerve function. Common forms include Magnesium Citrate (laxative, high bioavailability), Magnesium L-Threonate (blood-brain barrier penetration for cognitive support), and Magnesium Glycinate (calming for sleep and anxiety)."
    )
]

for q, gt in eval_queries:
    responses = {
        "Basic RAG": get_basic_rag_response(q),
        "Advanced RAG": get_advanced_rag_response(q),
        "LoRA FT": get_lora_ft_response(q),
        "DoRA FT": get_dora_ft_response(q)
    }
    evaluate_responses(q, gt, responses)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https:


Query: What are the primary benefits and forms of Magnesium?
Ground Truth: Magnesium is crucial for over 300 enzymatic reactions, ATP synthesis, and nerve function. Common forms include Magnesium Citrate (laxative, high bioavailability), Magnesium L-Threonate (blood-brain barrier penetration for cognitive support), and Magnesium Glycinate (calming for sleep and anxiety).

Basic RAG Response: Magnesium comes in several forms, each with distinct benefits and forms of use. Here’s a breakdown:
...
  -> ROUGE-L F1: 0.3307 | Semantic Sim: 0.7691

Advanced RAG Response: Magnesium comes in various forms, each offering different benefits. Here’s a summary:

- **Magnesium...
  -> ROUGE-L F1: 0.2313 | Semantic Sim: 0.8313

LoRA FT Response: Magnesium is crucial for over 300 enzymatic reactions, ATP synthesis, and nerve function. Common for...
  -> ROUGE-L F1: 0.6341 | Semantic Sim: 0.9196

DoRA FT Response: Magnesium is crucial for over 300 enzymatic reactions, ATP synthesis, and nerve function.